# Decision Tree Classification


## Assignment Explanation

### Objective
The objective of this assignment is to build a Decision Tree classification model and evaluate its performance on the heart disease dataset.

### Methodology
The dataset is explored for missing values, column types, and summary statistics. The target column is separated from the input features. Numerical columns are imputed with median values and categorical columns are imputed and encoded. The dataset is split into training and testing sets.

A Decision Tree classifier is trained, and hyperparameter tuning is performed using GridSearchCV. The best model is evaluated using classification metrics such as precision, recall, F1-score, and confusion matrix.

### Interpretation
A Decision Tree is easy to interpret because it splits the data using feature-based rules. The evaluation metrics show how well the model predicts the target class and whether it makes more false positives or false negatives.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)


In [22]:
%pip install scikit-learn
%pip install openpyxl
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df = pd.read_excel(
    r'C:\Users\lenovo\Downloads\Assignments\Decision Tree\Decision Tree\heart_disease.xlsx',
    sheet_name=0
)
xls = pd.ExcelFile(r'C:\Users\lenovo\Downloads\Assignments\Decision Tree\Decision Tree\heart_disease.xlsx')

for sheet in xls.sheet_names:
    df = pd.read_excel(r'C:\Users\lenovo\Downloads\Assignments\Decision Tree\Decision Tree\heart_disease.xlsx', sheet_name=sheet)
    print(f"\nSheet: {sheet}")
    print(df.shape)
    print(df.head())
df.head()

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

Sheet: Description
(12, 2)
        age                                       Age in years
0    Gender                       Gender ; Male - 1, Female -0
1        cp                                    Chest pain type
2  trestbps                             Resting blood pressure
3      chol                                cholesterol measure
4       fbs  (fasting blood sugar > 120 mg/dl) (1 = true; 0...

Sheet: Heart_disease
(908, 13)
   age   sex               cp  trestbps  chol    fbs         restecg  thalch  \
0   63  Male   typical angina       145   233   True  lv hypertrophy     150   
1   41  Male  atypical angina       135   203  False          normal     132   
2   57  Male     asymptomatic       140   192  False          normal     148   
3   52  Male   typical angina       118   186  False  lv hypertrophy     190   
4   57  Male     asymptomatic 

,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,thal,num
0,63,Male,typical angina,145,233,True,lv hypertrophy,150,False,2.3,downsloping,fixed defect,0
1,41,Male,atypical angina,135,203,False,normal,132,False,0.0,flat,fixed defect,0
2,57,Male,asymptomatic,140,192,False,normal,148,False,0.4,flat,fixed defect,0
3,52,Male,typical angina,118,186,False,lv hypertrophy,190,False,0.0,flat,fixed defect,0
4,57,Male,asymptomatic,110,201,False,normal,126,True,1.5,flat,fixed defect,0


In [23]:
df.info()
df.describe(include='all').T
df.head()
df = df.apply(pd.to_numeric, errors='ignore')
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 908 entries, 0 to 907
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       908 non-null    int64  
 1   sex       908 non-null    object 
 2   cp        908 non-null    object 
 3   trestbps  908 non-null    int64  
 4   chol      908 non-null    int64  
 5   fbs       908 non-null    bool   
 6   restecg   908 non-null    object 
 7   thalch    908 non-null    int64  
 8   exang     908 non-null    object 
 9   oldpeak   846 non-null    float64
 10  slope     908 non-null    object 
 11  thal      908 non-null    object 
 12  num       908 non-null    int64  
dtypes: bool(1), float64(1), int64(5), object(6)
memory usage: 86.1+ KB


(908, 13)

In [30]:
target = df.columns[-1]
X = df.drop(columns=[target])
y = df[target]

num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.columns.difference(num_cols)
cat_cols = X.select_dtypes(include=['object', 'bool']).columns
X[cat_cols] = X[cat_cols].astype(str)
preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [31]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

pipe = Pipeline([
    ('prep', preprocess),
    ('model', DecisionTreeClassifier(random_state=42))
])

params = {
    'model__max_depth': [3, 4, 5, None],
    'model__min_samples_split': [2, 5, 10],
    'model__criterion': ['gini', 'entropy']
}

grid = GridSearchCV(pipe, params, cv=cv, scoring='accuracy')

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

Best Params: {'model__criterion': 'entropy', 'model__max_depth': 4, 'model__min_samples_split': 10}


In [32]:
pred = grid.predict(X_test)
print(classification_report(y_test, pred))
confusion_matrix(y_test, pred)


              precision    recall  f1-score   support

           0       0.70      0.85      0.77        89
           1       0.38      0.44      0.41        48
           2       0.25      0.09      0.13        22
           3       0.09      0.06      0.07        17
           4       0.00      0.00      0.00         6

    accuracy                           0.55       182
   macro avg       0.29      0.29      0.28       182
weighted avg       0.48      0.55      0.51       182



array([[76, 11,  1,  1,  0],
       [20, 21,  2,  5,  0],
       [ 5, 11,  2,  4,  0],
       [ 5,  9,  2,  1,  0],
       [ 2,  3,  1,  0,  0]])